### 1. Initial Setup: Clean environment

In [ ]:
import os
import shutil

# Ensure the Python interpreter is in /content
os.chdir('/content')
print(f"Current working directory set to: {os.getcwd()}")

# Remove Colab's default sample_data directory
if os.path.exists('/content/sample_data'):
    shutil.rmtree('/content/sample_data')
    print("Removed /content/sample_data")

# Remove previously cloned Semantic-Correspondence repository if it exists
if os.path.exists('/content/Semantic-Correspondence'):
    shutil.rmtree('/content/Semantic-Correspondence')
    print("Removed /content/Semantic-Correspondence")

# Remove previously cloned data if it exists outside
if os.path.exists('/content/data'):
    shutil.rmtree('/content/data')
    print("Removed /content/data")

print("Environment cleaned.")


### 2. Clone the Semantic-Correspondence Repository

In [ ]:
import shutil
import os

%cd /content

# Remove previously cloned Semantic-Correspondence repository if it exists
if os.path.exists('/content/Semantic-Correspondence'):
    shutil.rmtree('/content/Semantic-Correspondence')
    print("Removed existing /content/Semantic-Correspondence directory.")

!git clone https://github.com/lucaosti/Semantic-Correspondence && echo "Repository cloned successfully." || echo "Error: Could not clone repository."


### 3. Change directory and Download Dataset

In [ ]:
# Change current directory to Semantic-Correspondence
%cd /content/Semantic-Correspondence

# Create the data directory if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')

# Download the dataset into the 'data/' directory
dataset_url = "https://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz"
dataset_path_in_repo = os.path.join('data', 'SPair-71k.tar.gz')

if not os.path.exists(dataset_path_in_repo):
    print(f"Downloading {dataset_url} to {dataset_path_in_repo}...")
    !wget -P data/ {dataset_url}
    print("Dataset downloaded.")
else:
    print("Dataset already exists in data/ directory.")


### 4. Google Drive — risultati intermedi e ripartenza

Esegui **dopo il clone e il download del dataset**, e **prima** di installare dipendenze / scaricare pesi, così `runs/` e `checkpoints/` non vengono creati solo su disco effimero.

Monta Drive e collega **`runs/`** e **`checkpoints/`** del repo a una cartella persistente: log, `pipeline_state.json`, `stage_events.jsonl`, export PCK, checkpoint di fine-tuning (`*_best.pt`, `*_resume.pt`) e pesi pre-addestrati scaricati.

- **Resume automatico:** con `pipeline_resume: true` nel `config.yaml`, alla ripartenza la pipeline salta gli stadi già registrati e il training riprende da `*_resume.pt` se presente (sempre nella cartella `checkpoints/` su Drive).
- **Stessa config:** dopo un nuovo clone, nella sezione 6 imposta `RESTORE_CONFIG_FROM_DRIVE = True` per riusare il `config.yaml` copiato su Drive (stessa impronta → resume coerente con `pipeline_state.json`).
- **Reset completo degli stadi** (senza cancellare i file su Drive): nella **sezione 6** imposta `START_FROM_SCRATCH = True` (la cella pipeline imposta automaticamente `SEMANTIC_CORRESPONDENCE_PIPELINE_RESET`). In alternativa, da shell: `export SEMANTIC_CORRESPONDENCE_PIPELINE_RESET=1`.
- **Attenzione:** se modifichi hyperparametri o toggle nel YAML rispetto al run precedente, l’impronta cambia e lo stato salvato viene ignorato.

Modifica `DRIVE_ARTIFACTS` se vuoi un percorso diverso sotto `MyDrive/`.

In [ ]:
import os
import shutil

REPO = "/content/Semantic-Correspondence"

# --- Personalizza: cartella su Google Drive che terrà runs/ e checkpoints/ ---
DRIVE_ARTIFACTS = "/content/drive/MyDrive/Colab/Semantic-Correspondence_artifacts"

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    USE_DRIVE = True
except ImportError:
    USE_DRIVE = False
    DRIVE_ARTIFACTS = os.path.join(REPO, "_persistent_artifacts")
    print("Ambiente non-Colab: uso cartella locale", DRIVE_ARTIFACTS)

os.makedirs(os.path.join(DRIVE_ARTIFACTS, "runs"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ARTIFACTS, "checkpoints"), exist_ok=True)


def _link_repo_dir_to_target(link_name: str, target_dir: str) -> None:
    """Replace repo/runs or repo/checkpoints with a symlink to Drive (or local mirror)."""
    link_path = os.path.join(REPO, link_name)
    if os.path.lexists(link_path):
        if os.path.islink(link_path):
            os.unlink(link_path)
        elif os.path.isdir(link_path):
            shutil.rmtree(link_path)
        else:
            os.remove(link_path)
    os.symlink(target_dir, link_path)


_link_repo_dir_to_target("runs", os.path.join(DRIVE_ARTIFACTS, "runs"))
_link_repo_dir_to_target("checkpoints", os.path.join(DRIVE_ARTIFACTS, "checkpoints"))

os.environ["SEMANTIC_DRIVE_ARTIFACTS"] = DRIVE_ARTIFACTS
print("Artefatti persistenti:", DRIVE_ARTIFACTS)
print("  →", REPO + "/runs")
print("  →", REPO + "/checkpoints")


### 5. Installazione dipendenze ed estrazione SPair-71k

Installa il pacchetto in editable mode, aggiunge il repo a `PYTHONPATH` ed estrae l'archivio scaricato nella cartella attesa da `verify_dataset` (`data/SPair-71k/`).

In [ ]:
import os
import sys
import tarfile

REPO = "/content/Semantic-Correspondence"
os.chdir(REPO)
sys.path.insert(0, REPO)

# Dipendenze progetto (include PyYAML, torch, …)
!pip install -q -e ".[notebook]"

os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

data_dir = os.path.join(REPO, "data")
archive = os.path.join(data_dir, "SPair-71k.tar.gz")
target = os.path.join(data_dir, "SPair-71k")

if os.path.isdir(target) and os.listdir(target):
    print("SPair-71k già presente:", target)
elif os.path.isfile(archive):
    print("Estrazione di", archive, "…")
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(path=data_dir)
    print("Estrazione completata.")
else:
    raise FileNotFoundError(
        "Manca data/SPair-71k.tar.gz — eseguire la cella di download (sezione 3)."
    )

layout = os.path.join(target, "Layout", "large", "test.txt")
if not os.path.isfile(layout):
    raise RuntimeError(f"Struttura SPair inattesa sotto {target!r} (manca {layout}).")
print("Dataset pronto:", target)



### 6. Pesi pre-addestrati e `config.yaml`

#### Parametri di modalità (in cima alla cella codice)

| Parametro | Valori | Effetto |
|-----------|--------|---------|
| `PIPELINE_RUN_MODE` | `"full"` | Verifica dataset, fine-tuning sui tre backbone, eval PCK, export metriche (come da template). |
| | `"finetune_only"` | **Solo** gli step `train_finetune` sui tre modelli: niente verify, eval, LoRA, pytest. Utile per concentrarsi sul training. |
| `START_FROM_SCRATCH` | `True` | Ignora `runs/pipeline_state.json` e **non** usa `*_resume.pt`: addestramento che riparte dai soli pesi pre-addestrati. |
| | `False` | Resume normale (`pipeline_resume` + stato su Drive). |
| `RESTORE_CONFIG_FROM_DRIVE` | `True` | Dopo un nuovo clone, ricarica `config.yaml` da Drive; i parametri sopra vengono **applicati** al file (sovrascrivono toggle e `pipeline_resume`). |

Scarica **tutti e tre** i checkpoint ufficiali (se non usi solo `RESTORE` senza rigenerare i path).

Se il download DINOv3 fallisse (es. 403), copia manualmente il `.pth` in `checkpoints/`.

**Hyperparametri** (epoche, `limit_pairs`, split, …): modifica `_build_fresh_config_dict()` nella cella codice, oppure `config.yaml` dopo la generazione (stesse chiavi lette da `python scripts/run_pipeline.py --config`).

In [ ]:
import os
import shutil

import yaml

REPO = "/content/Semantic-Correspondence"
os.chdir(REPO)

# =============================================================================
# PARAMETRI — modifica qui
# =============================================================================
# "full" = verify dataset + fine-tuning + eval PCK + export (come da template sotto)
# "finetune_only" = solo train_finetune sui tre backbone (nessun verify, eval, LoRA, pytest)
PIPELINE_RUN_MODE = "full"  # "full" | "finetune_only"

# True = riparti da zero: niente resume da pipeline_state né da *_resume.pt (solo pesi pre-addestrati)
# False = consenti resume (stato su Drive + checkpoint di ripresa se pipeline_resume nel YAML)
START_FROM_SCRATCH = False

# Dopo un nuovo clone: copia config da Drive, poi applica comunque PIPELINE_RUN_MODE e START_FROM_SCRATCH
RESTORE_CONFIG_FROM_DRIVE = False
# =============================================================================

DA = os.environ.get("SEMANTIC_DRIVE_ARTIFACTS")
DRIVE_CFG = os.path.join(DA, "config.yaml") if DA else None
cfg_path = os.path.join(REPO, "config.yaml")

os.environ["AML_START_FROM_SCRATCH"] = "1" if START_FROM_SCRATCH else ""
os.environ["AML_PIPELINE_RUN_MODE"] = PIPELINE_RUN_MODE


def _workflow_toggles_for_mode(mode: str) -> dict:
    if mode == "finetune_only":
        return {
            "run_verify_dataset": False,
            "train_finetune": [True, True, True],
            "train_lora": [False, False, False],
            "run_eval_baseline": [False, False, False],
            "run_eval_baseline_wsa": [False, False, False],
            "run_eval_finetuned_checkpoint": [False, False, False],
            "run_eval_lora_checkpoint": [False, False, False],
            "run_export_metrics_tables": False,
            "run_pytest": False,
            "print_jupyter_notebook_hint": False,
        }
    if mode != "full":
        raise ValueError("PIPELINE_RUN_MODE deve essere 'full' oppure 'finetune_only'")
    return {
        "run_verify_dataset": True,
        "train_finetune": [True, True, True],
        "train_lora": [False, False, False],
        "run_eval_baseline": [True, True, True],
        "run_eval_baseline_wsa": [False, False, False],
        "run_eval_finetuned_checkpoint": [True, True, True],
        "run_eval_lora_checkpoint": [False, False, False],
        "run_export_metrics_tables": True,
        "run_pytest": False,
        "print_jupyter_notebook_hint": True,
    }


def _apply_run_mode_to_cfg(cfg: dict) -> None:
    wt = cfg.setdefault("workflow_toggles", {})
    wt.update(_workflow_toggles_for_mode(PIPELINE_RUN_MODE))
    wt["pipeline_resume"] = not START_FROM_SCRATCH


def _build_fresh_config_dict() -> dict:
    d2 = f"{REPO}/checkpoints/dinov2_vitb14_pretrain.pth"
    d3 = f"{REPO}/checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
    sam = f"{REPO}/checkpoints/sam_vit_b_01ec64.pth"
    cfg = {
        "paths": {
            "repo_root": REPO,
            "spair_root": f"{REPO}/data/SPair-71k",
            "checkpoint_dir": "checkpoints",
            "output_dir": f"{REPO}/runs/notebook_exports",
        },
        "runtime": {
            "device": None,
            "num_workers": -1,
            "preprocess": "FIXED_RESIZE",
            "image_height": 784,
            "image_width": 784,
            "limit_pairs": 100,
            "eval_split": "val",
            "alphas": [0.05, 0.1, 0.15],
            "wsa_window": 5,
            "wsa_temperature": 1.0,
            "log_batch_interval": 100,
        },
        "finetune": {
            "last_blocks": 2,
            "epochs": 50,
            "patience": 10,
            "dinov2_weights": d2,
            "dinov3_weights": d3,
            "sam_checkpoint": sam,
        },
        "lora": {"rank": 16, "epochs": 5, "patience": 3},
        "workflow_toggles": {},
    }
    _apply_run_mode_to_cfg(cfg)
    return cfg


if RESTORE_CONFIG_FROM_DRIVE:
    # Non scarica i pesi: devono già essere in checkpoints/ (symlink Drive dalla sezione 4).
    if not DRIVE_CFG or not os.path.isfile(DRIVE_CFG):
        raise FileNotFoundError(
            "Manca config su Drive. Esegui prima un run con RESTORE_CONFIG_FROM_DRIVE = False "
            "oppure copia config.yaml in " + (DRIVE_CFG or "(imposta Drive, sezione 4)")
        )
    shutil.copy(DRIVE_CFG, cfg_path)
    with open(cfg_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if not isinstance(cfg, dict):
        raise TypeError("config.yaml da Drive non è un mapping YAML valido.")
    _apply_run_mode_to_cfg(cfg)
    with open(cfg_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)
    print("Config da Drive aggiornato (modalità + resume):", cfg_path)
else:
    !python scripts/download_pretrained_weights.py

    d2 = f"{REPO}/checkpoints/dinov2_vitb14_pretrain.pth"
    d3 = f"{REPO}/checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
    sam = f"{REPO}/checkpoints/sam_vit_b_01ec64.pth"
    for label, p in (("DINOv2", d2), ("DINOv3", d3), ("SAM", sam)):
        print(label, "OK" if os.path.isfile(p) else "MISSING", p)

    cfg = _build_fresh_config_dict()
    with open(cfg_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)
    print("Scritto:", cfg_path)

if DRIVE_CFG:
    shutil.copy(cfg_path, DRIVE_CFG)
    print("Backup config su Drive:", DRIVE_CFG)

print(
    f"Modalità: {PIPELINE_RUN_MODE} | Da zero: {START_FROM_SCRATCH} | "
    f"pipeline_resume nel file: {not START_FROM_SCRATCH}"
)



### 7. Esecuzione pipeline (`verify` → training opzionale → eval)

Esegue gli stadi abilitati in `config.yaml` (dipendono da **sezione 6**: `PIPELINE_RUN_MODE` e `START_FROM_SCRATCH`). **Riesegua la sezione 6** dopo ogni modifica a quei parametri prima di lanciare la pipeline. Log e stato sono in `runs/` (Drive, sezione 4).

- **`START_FROM_SCRATCH = True`**: questa cella imposta anche `SEMANTIC_CORRESPONDENCE_PIPELINE_RESET` così lo stato precedente in `runs/pipeline_state.json` viene azzerato per questa esecuzione.
- **`START_FROM_SCRATCH = False`** e `pipeline_resume: true` nel YAML: salta stadi già completati e il training può riprendere da `*_resume.pt` se presenti.


In [ ]:
import os

os.chdir("/content/Semantic-Correspondence")

# Allineato a START_FROM_SCRATCH (sezione 6): azzera pipeline_state.json per ripartire da zero
if os.environ.get("AML_START_FROM_SCRATCH", "").strip() == "1":
    os.environ["SEMANTIC_CORRESPONDENCE_PIPELINE_RESET"] = "1"
else:
    os.environ.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_RESET", None)

!python scripts/run_pipeline.py --config config.yaml



### 8. (Opzionale) Dashboard terminale sui log

Mostra lo stato della pipeline; su Colab l'output resta nel cell output.

In [ ]:
import os
os.chdir("/content/Semantic-Correspondence")
!bash scripts/start_dashboard.sh

